In [141]:
import pandas as pd

In [142]:
dataset = pd.read_csv("target_tabungan.csv")

In [143]:
dataset.head()

,tanggal,nama_target_id,nama_target,jumlah_target,nabung_harian,jumlah_terkumpul,progres,sisa_target,status_id,status
0,01/01/2022,0,Iphone 17 Pro Max,25000000,300000,300000,0.01,0.99,0,BELUM TERPENUHI
1,01/02/2022,0,Iphone 17 Pro Max,25000000,500000,800000,0.03,0.97,0,BELUM TERPENUHI
2,01/03/2022,0,Iphone 17 Pro Max,25000000,200000,1000000,0.04,0.96,0,BELUM TERPENUHI
3,01/04/2022,0,Iphone 17 Pro Max,25000000,500000,1500000,0.06,0.94,0,BELUM TERPENUHI
4,01/05/2022,0,Iphone 17 Pro Max,25000000,500000,2000000,0.08,0.92,0,BELUM TERPENUHI


In [144]:
dataset['tanggal'] = pd.to_datetime(dataset['tanggal'])

finish_dates = (
    dataset[dataset['status_id'] == 1]
    .groupby('nama_target_id')['tanggal']
    .max()
)

dataset['finish_date'] = dataset['nama_target_id'].map(finish_dates)

dataset['remaining_days'] = (
    dataset['finish_date'] - dataset['tanggal']
).dt.days

df = dataset[['tanggal', 'jumlah_target', 'nabung_harian', 'jumlah_terkumpul', 'status_id', 'remaining_days']].copy()

df['estimated_days_formula'] = (
    (df['jumlah_target'] - df['jumlah_terkumpul']) /
    df['nabung_harian']
)

df['tabungan_h1'] = (
    df['nabung_harian']
    .shift(1)
    .fillna(0)
)

df['tabungan_h2'] = (
    df['nabung_harian']
    .shift(2)
    .fillna(0)
)

df['tabungan_h3'] = (
    df['nabung_harian']
    .shift(3)
    .fillna(0)
)

df['avg_3d'] = (
    df[['tabungan_h1', 'tabungan_h2', 'tabungan_h3']]
    .mean(axis=1)
)

df['trend_3d'] = (
    df['tabungan_h1'] -
    df['tabungan_h3']
)

df

,tanggal,jumlah_target,nabung_harian,jumlah_terkumpul,status_id,remaining_days,estimated_days_formula,tabungan_h1,tabungan_h2,tabungan_h3,avg_3d,trend_3d
0,2022-01-01,25000000,300000,300000,0,106,82.333333,0.0,0.0,0.0,0.000000,0.0
1,2022-01-02,25000000,500000,800000,0,105,48.400000,300000.0,0.0,0.0,100000.000000,300000.0
2,2022-01-03,25000000,200000,1000000,0,104,120.000000,500000.0,300000.0,0.0,266666.666667,500000.0
3,2022-01-04,25000000,500000,1500000,0,103,47.000000,200000.0,500000.0,300000.0,333333.333333,-100000.0
4,2022-01-05,25000000,500000,2000000,0,102,46.000000,500000.0,200000.0,500000.0,400000.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1206,2025-04-21,8000000,300000,6750000,0,4,4.166667,500000.0,200000.0,50000.0,250000.000000,450000.0
1207,2025-04-22,8000000,200000,6950000,0,3,5.250000,300000.0,500000.0,200000.0,333333.333333,100000.0
1208,2025-04-23,8000000,100000,7050000,0,2,9.500000,200000.0,300000.0,500000.0,333333.333333,-300000.0
1209,2025-04-24,8000000,500000,7550000,0,1,0.900000,100000.0,200000.0,300000.0,200000.000000,-200000.0


In [145]:
from sklearn.model_selection import train_test_split

X = df[
    [
        'jumlah_target',
        'jumlah_terkumpul',
        'nabung_harian',
        'estimated_days_formula',
        'tabungan_h1',
        'tabungan_h2',
        'tabungan_h3',
        'avg_3d',
        'trend_3d'
    ]
]

y = df['remaining_days']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [146]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [147]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model

inputs = tf.keras.Input(shape=(X_train.shape[1],))

x = Dense(128, activation='relu')(inputs)
x = Dense(64, activation='relu')(x)
x = Dense(32, activation='relu')(x)

outputs = Dense(1)(x)

model = Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='mae',
    metrics=['mae']
)

In [148]:
class StopAtMAE(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        mae = logs.get('mae')

        if mae < 0.03:
            self.model.stop_training = True

In [149]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[StopAtMAE()]
)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 55.0427 - mae: 55.0427 - val_loss: 55.5926 - val_mae: 55.5926
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 50.1115 - mae: 50.1115 - val_loss: 46.6485 - val_mae: 46.6485
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 39.2031 - mae: 39.2031 - val_loss: 31.9552 - val_mae: 31.9552
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 27.0637 - mae: 27.0637 - val_loss: 22.1531 - val_mae: 22.1531
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 20.5243 - mae: 20.5243 - val_loss: 17.1496 - val_mae: 17.1496
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 16.4876 - mae: 16.4876 - val_loss: 12.8341 - val_mae: 12.8341
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 12.6870 - mae: 12.6870 - val_loss: 10.2534 - val_mae: 10.2534
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 10.4443 - mae: 10.4443 - val_loss: 8.8885 - val_mae: 8.8885
Epoch 9/100
25/25 ━━━━━━━━━━━━━━━

In [150]:
loss, mae = model.evaluate(X_test, y_test)

print("MAE:", mae)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.5505 - mae: 3.5505 
MAE: 3.5504939556121826


In [151]:
pred = model.predict(X_test)

comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': pred.flatten()
})

comparison.head(10)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


,Actual,Predicted
0,5,4.353425
1,131,131.213928
2,19,17.724783
3,129,123.692833
4,30,31.082136
5,55,51.795418
6,21,20.806288
7,57,58.886066
8,38,38.945076
9,40,43.293266


In [152]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, pred)

print("R2 Score:", r2)

R2 Score: 0.9827653765678406


In [153]:
model.export("SavedModel")

Saved artifact at 'SavedModel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 9), dtype=tf.float32, name='keras_tensor_25')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132522422454224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132522422463440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132522422456336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132522422455376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132522422457872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132522422462672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132522422453456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132522422460560: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [154]:
from google.colab import files
import shutil
import os

shutil.make_archive("SavedModel", 'zip', "SavedModel")
files.download("SavedModel.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [155]:
import joblib

joblib.dump(scaler, 'scaler.save')

['scaler.save']

Inference

In [158]:
def predict_saving_days(
    jumlah_target,
    jumlah_terkumpul,
    nabung_harian,
    tabungan_h1=0,
    tabungan_h2=0,
    tabungan_h3=0
):

    avg_3d = (
        tabungan_h1 +
        tabungan_h2 +
        tabungan_h3
    ) / 3

    if avg_3d == 0:
        avg_3d = nabung_harian

    trend_3d = (
        tabungan_h1 -
        tabungan_h3
    )

    sisa_target = max(
        jumlah_target - jumlah_terkumpul,
        0
    )

    estimated_days_formula = (
        sisa_target / avg_3d
    )

    if estimated_days_formula <= 7:
        return estimated_days_formula

    new_data = pd.DataFrame({
        'jumlah_target': [jumlah_target],
        'jumlah_terkumpul': [jumlah_terkumpul],
        'nabung_harian': [nabung_harian],
        'estimated_days_formula': [estimated_days_formula],
        'tabungan_h1': [tabungan_h1],
        'tabungan_h2': [tabungan_h2],
        'tabungan_h3': [tabungan_h3],
        'avg_3d': [avg_3d],
        'trend_3d': [trend_3d]
    })

    new_data_scaled = scaler.transform(new_data)

    ml_prediction = model.predict(
        new_data_scaled,
        verbose=0
    )[0][0]

    final_prediction = (
        estimated_days_formula * 0.5 +
        ml_prediction * 0.5
    )

    return max(final_prediction, 0)

In [159]:
test_cases = [
    {
        "name": "CASE 1 - Normal Stabil",
        "args": (
            30000000,
            15000000,
            200000,
            180000,
            200000,
            220000
        )
    },

    {
        "name": "CASE 2 - Makin Rajin",
        "args": (
            25000000,
            10000000,
            300000,
            200000,
            400000,
            700000
        )
    },

    {
        "name": "CASE 3 - Mulai Males",
        "args": (
            25000000,
            10000000,
            300000,
            300000,
            100000,
            0
        )
    },

    {
        "name": "CASE 4 - Hampir Selesai",
        "args": (
            10000000,
            9500000,
            100000,
            100000,
            100000,
            100000
        )
    },

    {
        "name": "CASE 5 - Super Rajin",
        "args": (
            50000000,
            30000000,
            1000000,
            700000,
            900000,
            1200000
        )
    },

    {
        "name": "CASE 6 - Cold Start",
        "args": (
            15000000,
            1000000,
            150000,
            0,
            0,
            0
        )
    },

    {
        "name": "CASE 7 - Extreme Kecil",
        "args": (
            20000000,
            1000000,
            10000,
            5000,
            10000,
            15000
        )
    }
]

for case in test_cases:
    prediction = predict_saving_days(*case["args"])

    print("=" * 50)
    print(case["name"])
    print(f"Prediction : {prediction:.2f} hari")

CASE 1 - Normal Stabil
Prediction : 70.14 hari
CASE 2 - Makin Rajin
Prediction : 53.18 hari
CASE 3 - Mulai Males
Prediction : 88.06 hari
CASE 4 - Hampir Selesai
Prediction : 5.00 hari
CASE 5 - Super Rajin
Prediction : 54.42 hari
CASE 6 - Cold Start
Prediction : 75.31 hari
CASE 7 - Extreme Kecil
Prediction : 1035.09 hari
